## Loading data

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil_types/plsda/soil_types.csv', sep=';') # local copy of Toledo and Guarapuava soil datasets
data = data_complete.loc[:, '1':'15']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

from modeling import pls_optimized

# performing PLS-DA with optimized latent variables
plsda_results = pls_optimized(Xcalclass_prep, 
                              ycalclass,
                              LVmax=2,
                              Xpred=Xpredclass_prep,
                              ypred=ypredclass,
                              aim='classification',
                              cv=10)
plsda_results[0]


# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:33,111 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:33,225 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

## Spectral cuts (domain knowledge)

In [2]:
# establishing spectral cuts based on expert knowledge of XRF spectra
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.34, 1.63),
('Si', 1.64, 1.86),
('P', 1.87, 2.19),
('S', 2.20, 2.44),
('background2', 2.45, 2.55),
('Rh L + Ar', 2.56, 3.21),
('K', 3.22, 3.53),
('Ca ka', 3.54, 3.84),
('Ca kb', 3.86, 4.14),
('background3', 4.15, 4.37),
('Ti ka', 4.38, 4.66),
('background4', 4.67, 4.75),
('Ti kb', 4.76, 5.12),
('Cr', 5.13, 5.77),
('Mn', 5.78, 6.13),
('Fe ka', 6.14, 6.80),
('Fe kb', 6.81, 7.30),
('background5', 7.31, 7.91),
('Cu', 7.92, 8.20),
('background6', 8.21, 10.69),
('Fe ka + Ti ka', 10.7, 11.14),
('background7', 11.15, 12.55),
('sum Fe' , 12.56, 13.1),
('background8', 13.11, 15.0)
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [3]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# # vamos agora extrair as variaveis mais importantes atraves do método SHAP
# import shap

# # Para PLSRegression, usamos KernelExplainer porque não há explainer dedicado muito rápido
# explainer_pls = shap.KernelExplainer(plsda_results[3].predict, Xcalclass_prep)
# shap_values_pls = explainer_pls(Xcalclass_prep)

# shap_global_importance = pd.DataFrame({
#     'energy': Xpredclass_prep.columns,
#     'Mean_Abs_SHAP': np.abs(shap_values_pls.values).mean(axis=0)}) # tomando a importancia global como a media dos valores absolutos dos valores SHAP para cada feature
# shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False, inplace=True)

# # vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
# energy_to_zone_shap = {}
# for zone_name, start, end in spectral_cuts:
#     for i in shap_global_importance['energy']:
#         i_float = float(i)
#         if start <= i_float <= end:
#             energy_to_zone_shap[i] = zone_name
# shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# # agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
# shap_unique_df = shap_global_importance.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
# shap_unique_df = shap_unique_df.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
# shap_unique_df.to_csv('shap_soil.csv', index=False, sep=';')
shap_unique_df = pd.read_csv('shap_soil_types.csv', sep=';') # loading previously saved shap_unique_df

# **Comparando com o bagging**

In [4]:
# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 42]

all_results = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=20,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    mi_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance',
        threshold=0.001, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results[seed] = {
        'bags_result': bags_result_seed,
        'mi_results_dict': mi_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")

# Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results[seed]['bags_result'],
        mi_results_dict=all_results[seed]['mi_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )

#     DG = exp.build_fold_predicate_graph(
#     bags_result=all_results[seed]['bags_result'],
#     mi_results_dict=all_results[seed]['mi_results_dict'],
#     predicates_df=predicates_quantiles[0],
#     random_state=42,
#     show_details=True,
#     normalize_weights=True,
#     weight_mode='ranking',
#     co_occurrence_matrix=co_occurrence_matrix_df,
#     apply_confidence_multiplier=True,
#     accumulate_cooccurrence_weights=True
# )

    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_by_seed[seed] = lrc_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_df_seed = lrc_by_seed[seed].rename(columns={'Node': f'Predicate_Seed_{seed}'})
    lrc_all_seeds_df = pd.concat([lrc_all_seeds_df, lrc_df_seed[[f'Predicate_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_unique_by_seed = {}
for seed, lrc_df in lrc_by_seed.items():
    lrc_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_unique_df = lrc_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_unique_by_seed[seed] = lrc_unique_df

lrc_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_11 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50
Bag_12 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 150 | Descartados: 50


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)



Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)



Processando LRC do grafo...


C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


,Predicate_Seed_0,Predicate_Seed_1,Predicate_Seed_42
0,Fe ka > -3.73,Fe ka > -3.73,Fe ka > -3.73
1,Fe ka <= 4.61,Fe ka <= 4.61,Fe ka <= 4.61
2,Fe kb > -1.37,Fe ka <= 2.41,Fe ka > -2.42
3,Fe ka <= 2.41,Fe ka > -2.42,Fe ka <= 2.41
4,Fe ka > -2.42,Fe kb > -1.37,Fe kb <= 1.68
...,...,...,...
146,background7 <= -0.12,Cr <= -0.17,S > 0.09
147,Class_A,background6 <= -0.13,background7 <= -0.12
148,Class_B,background5 <= 0.16,background2 <= -0.06
149,NaN,Class_A,Class_A


# Kennard-Stone + Round-Robin k-fold

## Duas Estratégias Disponíveis

### 1. Estratégia GLOBAL (`per_predicate=False`) - Original
- KS é aplicado **globalmente** em todas as amostras do dataset
- Distribui amostras via round-robin para k folds
- **Todos os predicados compartilham os mesmos folds**
- Predicados com cobertura < min_samples são eliminados

**Vantagens:**
- Consistência: mesmas amostras nos mesmos folds para todos os predicados
- Comparabilidade direta entre predicados
- Menor custo computacional (KS executado uma única vez)

**Desvantagens:**
- Predicados com baixa cobertura global podem ser eliminados
- A diversidade do KS é otimizada globalmente, não por predicado

---

### 2. Estratégia PER-PREDICATE (`per_predicate=True`) - Nova
- KS é aplicado **individualmente** para cada predicado
- Considera apenas as amostras que satisfazem cada predicado
- **Cada predicado tem seus próprios folds independentes**
- Resultados são combinados ao final

**Vantagens:**
- Maximiza representatividade dentro de cada predicado
- Mais amostras válidas por predicado (menos eliminações)
- Independência estatística entre predicados
- Diversidade otimizada para cada predicado individualmente

**Desvantagens:**
- Folds inconsistentes entre predicados (amostra X pode estar no Fold_1 para um predicado e Fold_3 para outro)
- Maior custo computacional (KS executado N vezes)
- Combinação de resultados requer cuidado na interpretação

**Solução técnica para KS unidimensional:**
- KS precisa de múltiplas variáveis para calcular distâncias
- Solução: adicionar o índice normalizado da amostra como segunda coluna
- Isso é determinístico e representa a "posição temporal" da amostra

In [5]:
import ks_folding as ksf

folds_result = ksf.kfold_predicates_roundrobin(
    zone_sums_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont,
    predicates_df=predicates_quantiles[0],
    k_folds=2,
    min_samples_ratio=0.001,  # 60% das amostras do fold
    verbose=True,
    per_predicate=False # escolhe entre fazer o fold por predicado ou globalmente (que faz o fold para todos os predicados juntos)
)

# Adiciona classe prevista (A/B) em cada DataFrame de predicado
for fold_name, pred_dict in folds_result.items():
    for rule, df_info in pred_dict.items():
        df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

mi_results_dict = exp.calculate_predicate_metrics(
    bags_result=folds_result,
    metric='covariance',
    threshold=0.001,
    #n_neighbors=5
)

max_len = max(len(mi_df['Predicate']) for mi_df in mi_results_dict.values())
padded_dict = {
    f'Predicate_{fold}': list(mi_df['Predicate']) + [None]*(max_len - len(mi_df['Predicate']))
    for fold, mi_df in mi_results_dict.items()
}
all_cov_results = pd.DataFrame(padded_dict)
all_cov_results

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:56,567 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:56,591 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.



Configuração KS + Round-Robin
Estratégia: GLOBAL (compartilhada)
Total de amostras: 501
Número de folds: 2
Amostras por fold (aprox.): 250
Mínimo de amostras por predicado: 2 (0% do fold)



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



=== Resumo (Estratégia GLOBAL - SEM estratificação) ===
Predicados totais: 200
Predicados eliminados (em pelo menos 1 fold): 0
Folds criados: 2
  Fold_1: 200 predicados válidos
  Fold_2: 200 predicados válidos
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001


,Predicate_Fold_1,Predicate_Fold_2
0,Fe ka > -3.73,Fe ka <= 4.61
1,Fe ka <= 4.61,Fe ka > -3.73
2,Fe ka > -2.42,Fe ka > -2.42
3,Fe ka <= 2.41,Fe ka <= 2.41
4,Fe kb > -1.37,Fe kb <= 1.68
...,...,...
177,background2 <= -0.09,background7 <= -0.15
178,background1 > 0.09,Rh L + Ar > 0.36
179,P <= -0.19,Cu > 0.15
180,S > 0.09,S <= -0.12


In [6]:
DG = exp.build_fold_predicate_graph(
    bags_result=folds_result,           # Resultado dos folds (KS + Round-Robin)
    mi_results_dict=mi_results_dict,    # Rankings de Covariância por fold
    predicates_df=predicates_quantiles[0],  # DataFrame com metadados dos predicados
    random_state=42,                    # Semente para reprodutibilidade
    show_details=True,                  # Mostra detalhes da resolução
    normalize_weights=True,            # False = peso inteiro | True = peso [1/k, 1]
    weight_mode='cooccurrence',         # 'ranking' ou 'cooccurrence'
    co_occurrence_matrix=co_occurrence_matrix_df,  # Necessário se weight_mode='cooccurrence'
    apply_confidence_multiplier=True,    # True = peso × score | False = só co-ocorrência
    accumulate_cooccurrence_weights=True  # Nova opção!
)
DG

# Calcula LRC para cada nó e compõe DataFrame
lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
# Seleciona apenas uma ocorrência por zona (maior LRC)
lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_ks_df

 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: ACUMULATIVA (soma valores da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 359

RESOLUÇÃO DE ARESTAS BIDIRECIONAIS
Total de pares bidirecionais encontrados: 6
Critério de desempate: PESO ACUMULADO (soma das co-ocorrências locais)

[Fe ka > -3.73 ↔ Fe ka <= 4.61]  EMPATE (peso=299.00)
  ✗ Removida (aleatório): Fe ka > -3.73 → Fe ka <= 4.61
  ✓ Mantida:  Fe ka <= 4.61 → Fe ka > -3.73

[Fe kb > -1.37 ↔ Fe kb <= 1.68]  EMPATE (peso=300.00)
  ✗ Removida (aleatório): Fe kb <= 1.68 → Fe kb > -1.37
  ✓ Mantida:  Fe kb > -1.37 → Fe kb <= 1.68

[Rh L + Ar > -0.26 ↔ Mn > -0.33]  EMPATE (peso=286.00)
  ✗ Removida (aleatório): Mn > -0.33 → Rh L + Ar > -0.26
  ✓ Mantida:  Rh L + Ar > -0.26 → Mn > -0.33

[Cr > -0.26 

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Cu > -0.22,3.008829,Cu,-0.22,>
1,Fe ka <= -3.73,2.637575,Fe ka,-3.73,<=
2,Fe ka > -3.73,2.479984,Fe ka,-3.73,>
3,Mn > -0.33,2.454457,Mn,-0.33,>
4,Si > -0.45,2.451985,Si,-0.45,>
...,...,...,...,...,...
191,Cr <= -0.26,0.546891,Cr,-0.26,<=
192,Rh L + Ar <= -0.26,0.342921,Rh L + Ar,-0.26,<=
193,background6 <= -0.13,0.316532,background6,-0.13,<=
194,Class_A,0.000000,None,None,None


# **Variando o numero de folds - modo cooccurrence**

In [7]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_cooc = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_cooc[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='cooccurrence',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=False
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:58,106 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:58,110 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 501
Número de folds: 2
Amostras por fold (aprox.): 250
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:58,348 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:06:58,352 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 200
Predicados válidos: 200
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -0.10': 108 amostras, folds: [54, 54]
  'background1 > -0.10': 393 amostras, folds: [197, 196]
  'background1 <= -0.08': 190 amostras, folds: [95, 95]
  'background1 > -0.08': 311 amostras, folds: [156, 155]
  'background1 <= 0.09': 297 amostras, folds: [149, 148]
  ... e mais 195 predicados

Predicados por fold:
  Fold_1: 200 predicados
  Fold_2: 200 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: Matriz de co-ocorrência global
Sub-estratégia: NÃO-ACUMULATIVA (peso fixo da matriz)

Folds processados: 2
Arestas criadas (antes de resolver bidirecionais): 358

RESOLUÇÃO DE

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:05,538 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:05,540 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarni

...

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:30,787 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:30,789 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarni


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 501
Número de folds: 6
Amostras por fold (aprox.): 83
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:31,019 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:31,021 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 200
Predicados válidos: 200
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -0.10': 108 amostras, folds: [18, 18, 18, 18, 18, 18]
  'background1 > -0.10': 393 amostras, folds: [66, 66, 66, 65, 65, 65]
  'background1 <= -0.08': 190 amostras, folds: [32, 32, 32, 32, 31, 31]
  'background1 > -0.08': 311 amostras, folds: [52, 52, 52, 52, 52, 51]
  'background1 <= 0.09': 297 amostras, folds: [50, 50, 50, 49, 49, 49]
  ... e mais 195 predicados

Predicados por fold:
  Fold_1: 200 predicados
  Fold_2: 200 predicados
  Fold_3: 200 predicados
  Fold_4: 200 predicados
  Fold_5: 200 predicados
  Fold_6: 200 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: normalize_weights=True ignorado em weight_mode='cooccurrence'
   (Pesos de co-ocorrência representam contagens reais de amostras)

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: COOCCURRENCE
Fonte: M

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\networkx\algorithms\centrality\reaching.py:193: RuntimeWarning: divide by zero encountered in scalar divide
  return total_weight / d.get(weight, 1)


In [8]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
     features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values

# vamos exportar o df features_importance para um arquivo excel onde vamos nomear a sheet de acordo com as comparacoes feitas
features_importance.to_excel('features_importance_soil_types.xlsx', index=False, sheet_name='KS_fold_cooc')
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe kb,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Fe kb,Ca ka,Ca ka,sum Fe,Fe ka,Fe kb,Fe kb,sum Fe,Fe kb,Fe kb,Fe kb
2,Ti ka,Ti ka,Ti ka,Ca ka,Mn,Ti ka,sum Fe,Ti kb,Mn,Mn,Cr
3,Ca ka,Fe kb,Mn,Fe kb,Ti kb,Mn,Ti ka,Fe kb,Ti kb,sum Fe,background4
4,Mn,Mn,Fe kb,Mn,Ti ka,Ti kb,background4,background4,sum Fe,Ti ka,Mn
5,sum Fe,Al,Al,Ti ka,background4,sum Fe,Mn,Mn,Cr,background4,Si
6,Ti kb,Rh L + Ar,background6,Rh L + Ar,Cr,K,Rh L + Ar,Cr,Si,Ca ka,Ti kb
7,Al,Ca kb,Fe ka + Ti ka,Si,Ca ka,Si,Cr,K,background4,Rh L + Ar,sum Fe
8,Si,K,background7,Cr,sum Fe,background8,Si,Ti ka,Rh L + Ar,Cr,Rh L + Ar
9,background4,Ti kb,background8,Ti kb,Ca kb,Rh L + Ar,Ti kb,Cu,Cu,Ti kb,Ti ka


In [9]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_excel('rbo_soil_types.xlsx', index=False, sheet_name='KS_fold_cooc')
rbo_results

,Reference,Method,RBO_Score
4,Vip,LRC_kfold_4,0.904524
5,Vip,LRC_kfold_5,0.832277
8,Vip,LRC_Seed_1,0.829088
7,Vip,LRC_Seed_0,0.796938
9,Vip,LRC_Seed_42,0.784703
0,Vip,Reg_coef,0.783692
1,Vip,Shap,0.756756
2,Vip,LRC_kfold_2,0.711138
6,Vip,LRC_kfold_6,0.628532
3,Vip,LRC_kfold_3,0.517564


# **Variando o numero de folds - modo ranking**

In [10]:
# Loop para gerar múltiplos grafos e DataFrames LRC para diferentes valores de k_folds

folds_list = [2, 3, 4, 5, 6]  # Exemplo de diferentes valores de k_folds

graphs_ks_by_fold = {}
lrc_ks_df_by_fold = {}
lrc_ks_unique_df_by_fold = {}
all_fold_results_rank = {}

for k_folds in folds_list:
    print(f"\n{'='*70}")
    print(f"Processando k_folds: {k_folds}")
    print(f"{'='*70}\n")

    # Geração dos folds
    folds_result = ksf.kfold_predicates_roundrobin(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_pred_cont,
        predicates_df=predicates_quantiles[0],
        k_folds=k_folds,
        min_samples_ratio=0.001,
        verbose=True,
        per_predicate=True
    )

    # Adiciona classe prevista (A/B) em cada DataFrame de predicado
    for fold_name, pred_dict in folds_result.items():
        for rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B')

    # Calcula métricas de covariância
    mi_results_dict = exp.calculate_predicate_metrics(
        bags_result=folds_result,
        metric='covariance',
        threshold=0.001,
    )

        # Salvar no dicionário principal
    all_fold_results_rank[k_folds] = {
        'cov_results_dict': mi_results_dict
    }

    # Constrói o grafo
    DG = exp.build_fold_predicate_graph(
        bags_result=folds_result,
        mi_results_dict=mi_results_dict,
        predicates_df=predicates_quantiles[0],
        random_state=42,
        show_details=True,
        normalize_weights=True,
        weight_mode='ranking',
        co_occurrence_matrix=co_occurrence_matrix_df,
        apply_confidence_multiplier=True,
        accumulate_cooccurrence_weights=True
    )

    # Calcula LRC para cada nó e compõe DataFrame
    lrc_ks_df = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_ks_unique_df = lrc_ks_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)

    # Salva nos dicionários
    graphs_ks_by_fold[k_folds] = DG
    lrc_ks_df_by_fold[k_folds] = lrc_ks_df
    lrc_ks_unique_df_by_fold[k_folds] = lrc_ks_unique_df

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:39,999 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:40,001 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 2

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 501
Número de folds: 2
Amostras por fold (aprox.): 250
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:40,249 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:40,259 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 200
Predicados válidos: 200
Predicados eliminados: 0
Folds criados: 2

Estatísticas por predicado válido:
  'background1 <= -0.10': 108 amostras, folds: [54, 54]
  'background1 > -0.10': 393 amostras, folds: [197, 196]
  'background1 <= -0.08': 190 amostras, folds: [95, 95]
  'background1 > -0.08': 311 amostras, folds: [156, 155]
  'background1 <= 0.09': 297 amostras, folds: [149, 148]
  ... e mais 195 predicados

Predicados por fold:
  Fold_1: 200 predicados
  Fold_2: 200 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrências só se aplica ao modo 'cooccurrence')

CONSTRUÇÃO DO GRAFO DE PREDICADOS
Modo de peso: RANKING

Folds processados: 2
Arestas cri

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:47,412 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:07:47,414 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

...


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 200
Predicados válidos: 200
Predicados eliminados: 0
Folds criados: 5

Estatísticas por predicado válido:
  'background1 <= -0.10': 108 amostras, folds: [22, 22, 22, 21, 21]
  'background1 > -0.10': 393 amostras, folds: [79, 79, 79, 78, 78]
  'background1 <= -0.08': 190 amostras, folds: [38, 38, 38, 38, 38]
  'background1 > -0.08': 311 amostras, folds: [63, 62, 62, 62, 62]
  'background1 <= 0.09': 297 amostras, folds: [60, 60, 59, 59, 59]
  ... e mais 195 predicados

Predicados por fold:
  Fold_1: 200 predicados
  Fold_2: 200 predicados
  Fold_3: 200 predicados
  Fold_4: 200 predicados
  Fold_5: 200 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_mode='ranking'
   (Acumulação de co-ocorrência

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:08:10,490 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:08:10,492 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Processando k_folds: 6

Configuração KS + Round-Robin
Estratégia: PER-PREDICATE (individual)
Total de amostras: 501
Número de folds: 6
Amostras por fold (aprox.): 83
Mínimo de amostras por predicado: 2 (0% do fold)

Iniciando estratégia PER-PREDICATE...
KS será aplicado individualmente para cada predicado.



C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:08:10,711 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-01-19 09:08:10,714 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur


Resumo (Estratégia PER-PREDICATE)
Predicados totais: 200
Predicados válidos: 200
Predicados eliminados: 0
Folds criados: 6

Estatísticas por predicado válido:
  'background1 <= -0.10': 108 amostras, folds: [18, 18, 18, 18, 18, 18]
  'background1 > -0.10': 393 amostras, folds: [66, 66, 66, 65, 65, 65]
  'background1 <= -0.08': 190 amostras, folds: [32, 32, 32, 32, 31, 31]
  'background1 > -0.08': 311 amostras, folds: [52, 52, 52, 52, 52, 51]
  'background1 <= 0.09': 297 amostras, folds: [50, 50, 50, 49, 49, 49]
  ... e mais 195 predicados

Predicados por fold:
  Fold_1: 200 predicados
  Fold_2: 200 predicados
  Fold_3: 200 predicados
  Fold_4: 200 predicados
  Fold_5: 200 predicados
  Fold_6: 200 predicados
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.001
 AVISO: apply_confidence_multiplier=True ignorado em weight_mode='ranking'
   (Multiplicador de confiança só se aplica ao modo 'cooccurrence')
 AVISO: accumulate_cooccurrence_weights=True ignorado em weight_m

In [11]:
features_importance = pd.DataFrame({
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
    })

for k_folds, lrc_unique_df in lrc_ks_unique_df_by_fold.items():
    features_importance[f'LRC_kfold_{k_folds}'] = lrc_unique_df['Zone'].iloc[:10].values

for seed, lrc_unique_df in lrc_unique_by_seed.items():
    features_importance[f'LRC_Seed_{seed}'] = lrc_unique_df['Zone'].iloc[:10].values
    
with pd.ExcelWriter('features_importance_soil_types.xlsx', mode='a', engine='openpyxl') as writer:
    features_importance.to_excel(writer, sheet_name='KS_fold_rank', index=False)
features_importance

,Vip,Reg_coef,Shap,LRC_kfold_2,LRC_kfold_3,LRC_kfold_4,LRC_kfold_5,LRC_kfold_6,LRC_Seed_0,LRC_Seed_1,LRC_Seed_42
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe kb,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Fe kb,Ca ka,Ca ka,Fe kb,Fe kb,Fe ka,Fe kb,sum Fe,Fe kb,Fe kb,Fe kb
2,Ti ka,Ti ka,Ti ka,sum Fe,Ca ka,Ti ka,sum Fe,Mn,Mn,Mn,Cr
3,Ca ka,Fe kb,Mn,Ca ka,Mn,Ca ka,Ti ka,Fe kb,Ti kb,sum Fe,background4
4,Mn,Mn,Fe kb,Cu,background4,Ti kb,Mn,Ca ka,sum Fe,Ti ka,Mn
5,sum Fe,Al,Al,Mn,sum Fe,background8,background4,Ti kb,Cr,background4,Si
6,Ti kb,Rh L + Ar,background6,Ti ka,Si,sum Fe,Ca ka,background4,Si,Ca ka,Ti kb
7,Al,Ca kb,Fe ka + Ti ka,Rh L + Ar,Ti ka,Al,Rh L + Ar,Cu,background4,Rh L + Ar,sum Fe
8,Si,K,background7,Cr,Ca kb,Mn,Si,Cr,Rh L + Ar,Cr,Rh L + Ar
9,background4,Ti kb,background8,Ti kb,Rh L + Ar,Cu,Cr,K,Cu,Ti kb,Ti ka


In [12]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'LRC_kfold_{fold}' for fold in folds_list] + [f'LRC_Seed_{seed}' for seed in random_seeds]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)

with pd.ExcelWriter('rbo_soil_types.xlsx', mode='a', engine='openpyxl') as writer:
    rbo_results.to_excel(writer, sheet_name='KS_fold_rank', index=False)
rbo_results

,Reference,Method,RBO_Score
5,Vip,LRC_kfold_5,0.856735
3,Vip,LRC_kfold_3,0.851693
2,Vip,LRC_kfold_2,0.839197
8,Vip,LRC_Seed_1,0.829088
7,Vip,LRC_Seed_0,0.796938
9,Vip,LRC_Seed_42,0.784703
0,Vip,Reg_coef,0.783692
1,Vip,Shap,0.756756
6,Vip,LRC_kfold_6,0.673878
4,Vip,LRC_kfold_4,0.628066


# **Comparando com rankings medios**

In [26]:
ranking_predicate_mean_seeds = {}
ranking_predicate_mean_unique_seeds = {}

for seed in random_seeds:
    mi_results_dict_seed = all_results[seed]['mi_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_seed, 
        return_unique_zones=True
    )
    ranking_predicate_mean_seeds[seed] = ranking_predicate_mean
    ranking_predicate_mean_unique_seeds[seed] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [27]:
ranking_predicate_mean_folds = {}
ranking_predicate_mean_unique_folds = {}

for fold in folds_list:
    mi_results_dict_fold = all_fold_results_cooc[fold]['cov_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_fold, 
        return_unique_zones=True
    )
    ranking_predicate_mean_folds[fold] = ranking_predicate_mean
    ranking_predicate_mean_unique_folds[fold] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [28]:
# Build the dictionary step by step to avoid mixing comprehension and static entries
features_dict = {
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
}
# Add the predicate rankings from seeds
for seed, ranking_df in ranking_predicate_mean_unique_seeds.items():
    features_dict[f'Cov_Mean_Seed_{seed}'] = ranking_df['Zone'].iloc[:10].values
# Add the predicate rankings from folds
for fold, ranking_df in ranking_predicate_mean_unique_folds.items():
    features_dict[f'Cov_Mean_Fold_{fold}'] = ranking_df['Zone'].iloc[:10].values
features_importance = pd.DataFrame(features_dict)
features_importance

,Vip,Reg_coef,Shap,Cov_Mean_Seed_0,Cov_Mean_Seed_1,Cov_Mean_Seed_42,Cov_Mean_Fold_2,Cov_Mean_Fold_3,Cov_Mean_Fold_4,Cov_Mean_Fold_5,Cov_Mean_Fold_6
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Fe kb,Ca ka,Ca ka,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb
2,Ti ka,Ti ka,Ti ka,Mn,Mn,Mn,Mn,Ca ka,Mn,Mn,Mn
3,Ca ka,Fe kb,Mn,sum Fe,Ti ka,sum Fe,Ti ka,Mn,sum Fe,sum Fe,sum Fe
4,Mn,Mn,Fe kb,Ti ka,sum Fe,Ti ka,sum Fe,sum Fe,Ti ka,Ti ka,Ti ka
5,sum Fe,Al,Al,Ca ka,Rh L + Ar,Ca ka,Ca ka,Ti ka,Rh L + Ar,Rh L + Ar,Ca ka
6,Ti kb,Rh L + Ar,background6,Rh L + Ar,Ca ka,Rh L + Ar,Rh L + Ar,Rh L + Ar,Ti kb,Ti kb,Rh L + Ar
7,Al,Ca kb,Fe ka + Ti ka,Ti kb,Ti kb,Si,Si,Si,Si,Si,Ti kb
8,Si,K,background7,Si,Si,Ti kb,Ti kb,Ti kb,Cr,Ca ka,Si
9,background4,Ti kb,background8,Cr,Cr,Cr,Cr,Cr,Ca ka,Cr,background4


In [29]:
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'Cov_Mean_Seed_{seed}' for seed in random_seeds] + [f'Cov_Mean_Fold_{fold}' for fold in folds_list]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
5,Vip,Cov_Mean_Fold_2,0.867060
6,Vip,Cov_Mean_Fold_3,0.867060
3,Vip,Cov_Mean_Seed_1,0.861745
9,Vip,Cov_Mean_Fold_6,0.845634
2,Vip,Cov_Mean_Seed_0,0.844423
4,Vip,Cov_Mean_Seed_42,0.841335
8,Vip,Cov_Mean_Fold_5,0.832931
7,Vip,Cov_Mean_Fold_4,0.831010
0,Vip,Reg_coef,0.783692
1,Vip,Shap,0.756756


## modo ranking

In [30]:
ranking_predicate_mean_folds = {}
ranking_predicate_mean_unique_folds = {}

for fold in folds_list:
    mi_results_dict_fold = all_fold_results_rank[fold]['cov_results_dict']
    ranking_predicate_mean, ranking_predicate_mean_unique = exp.calculate_predicate_ranking_mean(
        mi_results_dict_fold, 
        return_unique_zones=True
    )
    ranking_predicate_mean_folds[fold] = ranking_predicate_mean
    ranking_predicate_mean_unique_folds[fold] = ranking_predicate_mean_unique
# Exibindo resultados para uma semente específica

In [31]:
# Build the dictionary step by step to avoid mixing comprehension and static entries
features_dict = {
    'Vip' : vip_scores_unique_df['Zone'].iloc[:10].values,
    'Reg_coef' : reg_vet_unique_df['Zone'].iloc[:10].values,
    'Shap' : shap_unique_df['Zone'].iloc[:10].values
}
# Add the predicate rankings from seeds
for seed, ranking_df in ranking_predicate_mean_unique_seeds.items():
    features_dict[f'Cov_Mean_Seed_{seed}'] = ranking_df['Zone'].iloc[:10].values
# Add the predicate rankings from folds
for fold, ranking_df in ranking_predicate_mean_unique_folds.items():
    features_dict[f'Cov_Mean_Fold_{fold}'] = ranking_df['Zone'].iloc[:10].values
features_importance = pd.DataFrame(features_dict)
features_importance

,Vip,Reg_coef,Shap,Cov_Mean_Seed_0,Cov_Mean_Seed_1,Cov_Mean_Seed_42,Cov_Mean_Fold_2,Cov_Mean_Fold_3,Cov_Mean_Fold_4,Cov_Mean_Fold_5,Cov_Mean_Fold_6
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Fe kb,Ca ka,Ca ka,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb,Fe kb
2,Ti ka,Ti ka,Ti ka,Mn,Mn,Mn,Mn,Ca ka,Mn,Mn,Mn
3,Ca ka,Fe kb,Mn,sum Fe,Ti ka,sum Fe,Ti ka,Mn,sum Fe,sum Fe,sum Fe
4,Mn,Mn,Fe kb,Ti ka,sum Fe,Ti ka,sum Fe,sum Fe,Ti ka,Ti ka,Ti ka
5,sum Fe,Al,Al,Ca ka,Rh L + Ar,Ca ka,Ca ka,Ti ka,Rh L + Ar,Rh L + Ar,Ca ka
6,Ti kb,Rh L + Ar,background6,Rh L + Ar,Ca ka,Rh L + Ar,Rh L + Ar,Rh L + Ar,Ti kb,Ti kb,Rh L + Ar
7,Al,Ca kb,Fe ka + Ti ka,Ti kb,Ti kb,Si,Si,Si,Si,Si,Ti kb
8,Si,K,background7,Si,Si,Ti kb,Ti kb,Ti kb,Cr,Ca ka,Si
9,background4,Ti kb,background8,Cr,Cr,Cr,Cr,Cr,Ca ka,Cr,background4


In [32]:
import rbo
rbo_results = {}
reference_list = features_importance['Vip'].tolist()
methods = ['Reg_coef', 'Shap'] + [f'Cov_Mean_Seed_{seed}' for seed in random_seeds] + [f'Cov_Mean_Fold_{fold}' for fold in folds_list]
for method in methods:
    compare_list = features_importance[method].tolist()
    score = rbo.RankingSimilarity(reference_list, compare_list).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results

,Reference,Method,RBO_Score
5,Vip,Cov_Mean_Fold_2,0.867060
6,Vip,Cov_Mean_Fold_3,0.867060
3,Vip,Cov_Mean_Seed_1,0.861745
9,Vip,Cov_Mean_Fold_6,0.845634
2,Vip,Cov_Mean_Seed_0,0.844423
4,Vip,Cov_Mean_Seed_42,0.841335
8,Vip,Cov_Mean_Fold_5,0.832931
7,Vip,Cov_Mean_Fold_4,0.831010
0,Vip,Reg_coef,0.783692
1,Vip,Shap,0.756756
